![Platform+](images/banner.png)

# LAB 1: NAVIGATE + CREATE OBJECTS
_Hands-on Snowflake for data practitioners_

> **Persona track: Platform Admin** — same lab, your angle.

👉 **In this lesson, we'll find your way around Snowflake and create your first objects.**

📚 **By the end you'll be able to:**
- Set and confirm your query context
- Explore the object hierarchy
- Create your own schema, table, and row in your own database
- Run a metadata inventory of objects and column types
- Check who already has access to sensitive tables
- Verify warehouse governance settings

**Time:** ~20 min
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to a compute service (first time):**

> 💡 **Read all four steps before you click Connect.** The connect dialog opens on top of the notebook and covers these instructions while it's open — skim them first; to re-read, cancel the dialog or scroll the notebook down a little before reopening it.

1. Look at the top bar of the notebook — if no service is connected, click **Connect**.
2. In the connect dialog, either pick an existing service or fill in the details and click **Create and connect** at the bottom to provision one (this can take up to a minute or two).
3. **Wait until the status shows Connected (green).** You can't set your context or run any cell until it does.
4. Now set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.) Run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.

> 💡 **Opening the documentation links:** Throughout this course, doc links open in the *same* tab and can hit a Snowsight "Application error." To read a linked page, **right-click the link and choose "Open link in new tab"** (or Cmd/Ctrl-click it).

## Session Variables
The cell below calls `get_active_session()` — a Snowpark function that gives Python the **same Snowflake session your SQL cells use**. From it we read your username and build a few variable *names* (`my_db`, `my_wh`, `my_role`) that follow the course naming convention.

> ⚠️ At this point these are just Python strings — building them doesn't activate anything. The **next cell (SQL)** is what actually runs `USE ROLE` / `USE WAREHOUSE` to set your session context, so the role/warehouse picker at the top-right won't update until that SQL runs.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)

## Set Up Your Workspace
First, set up your **personal workspace**. The cell below switches to your own role, creates and uses your own warehouse, and creates your own database for what you'll build in this lab. Every object is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Your personal database + the schema(s) this lab writes to
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.SANDBOX;
CREATE SCHEMA IF NOT EXISTS {{my_db}}.STAGING;
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · Where Am I?
Every query runs in a **context**: role · warehouse · database · schema. You just set yours in the `provision_workspace` cell above — let's confirm it.

These are **context functions** — each returns one piece of your current session: [`CURRENT_USER()`](https://docs.snowflake.com/en/sql-reference/functions/current_user), [`CURRENT_ROLE()`](https://docs.snowflake.com/en/sql-reference/functions/current_role), [`CURRENT_WAREHOUSE()`](https://docs.snowflake.com/en/sql-reference/functions/current_warehouse), [`CURRENT_DATABASE()`](https://docs.snowflake.com/en/sql-reference/functions/current_database), and [`CURRENT_SCHEMA()`](https://docs.snowflake.com/en/sql-reference/functions/current_schema). They're handy any time you're unsure where a query will run. See the [context functions reference](https://docs.snowflake.com/en/sql-reference/functions-context).

> 💡 **Why this matters for admins:** Every security audit starts with "who ran what, as which role, on which warehouse?" Context functions are how you verify that your own session — and your users' sessions — are running with least privilege.

👉 Run the cell to see yours.

In [ ]:
-- Confirm your context
SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE(), CURRENT_SCHEMA();

> 🔎 **Review the output.** You should see your personal role (ends in `_RL`) and your own warehouse, pointed at `PLATPLUS_DB.ROASTERY_SALES` for reading. You *read* from the shared data here, but *build* in your own database.

## 2 · What's Here?
Objects live in a hierarchy: **account → database → schema → table/view**. Explore it with [`SHOW`](https://docs.snowflake.com/en/sql-reference/sql/show) commands.

👉 Run the next two cells.

In [ ]:
-- The schemas (folders of objects) inside the shared database
SHOW SCHEMAS IN DATABASE PLATPLUS_DB;

In [ ]:
-- The tables inside the ROASTERY_SALES schema
SHOW TABLES IN SCHEMA PLATPLUS_DB.ROASTERY_SALES;

Now peek at the actual rows. You'll notice `STORES` is short — one row per café — while `ORDERS` is long, a row for every purchase. 👉 Run each cell and look at the columns.

In [ ]:
-- STORES: one row per Roastery & Co. location (read-only shared data)
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.STORES;

In [ ]:
-- ORDERS: one row per order (limit to 10 just to peek)
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS LIMIT 10;

> 🔎 **Look at the columns in both tables.** Notice that `ORDERS` has a `STORE_ID`, and `STORES` also has a `STORE_ID`. That shared column is the **link** between the two tables.
>
> `ORDERS` tells you *how much* was sold, but only as a store **id** number. `STORES` tells you each store's **name** and city. Neither table has the whole picture on its own.
>
> For anyone newer to SQL: combining two tables on a shared column like this is called a **join** — it lets you answer a question that spans both, e.g. *"total sales by store **name**"*. You'll use a join in the next topic; for now, just spot the connection.
>
> **Your turn to think:** which column links `ORDERS` to `STORES`?

### See it. 📊
**Plotly** is a charting library pre-installed in Snowflake Notebooks — pass it a table and specify which columns map to x and y; it renders an interactive bar chart you can hover over. A quick look at sales by store — your first inline chart.

> 💡 **Why this matters for admins:** Visual spot-checks help you verify data before granting broader access — if a chart looks wrong, you don't want analysts drawing conclusions from it.

In [ ]:
import plotly.express as px
from snowflake.snowpark.context import get_active_session
session = get_active_session()
df = session.sql("""SELECT store_name, total_sales FROM PLATPLUS_DB.ROASTERY_SALES.SALES_BY_STORE ORDER BY total_sales DESC""").to_pandas()
fig = px.bar(df, x="STORE_NAME", y="TOTAL_SALES", title="Sales by store",
             color_discrete_sequence=["#29B5E8"])
fig.update_layout(xaxis_title="Store Name", yaxis_title="Total Sales")
fig

## 3 · Make Something
Now build something of your own in **your own** database — a personal **coffee tasting log**. You own what you create here. See [`CREATE TABLE`](https://docs.snowflake.com/en/sql-reference/sql/create-table).

👉 Run the cell to create the table and your first tasting note.

> 📌 **Note:** You can't break the shared `PLATPLUS_DB` — it's read-only to you. Your own database (named after your username) is your space to experiment in.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.SANDBOX.MY_TASTING_NOTES (
  tasting_id NUMBER, coffee STRING, rating NUMBER, notes STRING, tasted_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP());
INSERT INTO {{my_db}}.SANDBOX.MY_TASTING_NOTES (tasting_id, coffee, rating, notes)
  VALUES (1, 'Ethiopia Yirgacheffe', 5, 'Bright and floral — my first object on Snowflake');

> 👉 **Your turn.** Make this table yours — edit the values below to a coffee *you* like and a rating from 1–5, then run the cell.

In [ ]:
-- YOUR TURN: change the coffee, rating (1-5), and note to your own, then run
INSERT INTO {{my_db}}.SANDBOX.MY_TASTING_NOTES (tasting_id, coffee, rating, notes)
  VALUES (2, 'YOUR COFFEE HERE', 5, 'What did you think of it?');

Next, read your table back to confirm both rows landed — the seeded example plus the one you just added.

In [ ]:
-- See both rows: the seeded example and the one you just added
SELECT * FROM {{my_db}}.SANDBOX.MY_TASTING_NOTES ORDER BY tasting_id;

> 🔎 **Review the output.** Two rows now — the example plus your own. You created a table and inserted your own data into it.

### Your angle — Platform Admin.

The DA explores data. The DE transforms it. Your job is different: **know your environment before you can govern it**. The questions every admin answers before touching access controls or cost policies:

1. What objects exist, how large are they, and which columns look sensitive?
2. Who already has access to the sensitive ones?
3. Is my compute resource configured to avoid runaway spend?

Then: document what you've created so the next admin doesn't have to reverse-engineer it.

#### Step 1 · Know your inventory

`INFORMATION_SCHEMA` is the admin's primary metadata tool — it gives you structured information about every object without reading the data itself. Start with table sizes, then go deeper into columns.

> 💡 **What is `INFORMATION_SCHEMA`?** Every Snowflake database includes a built-in, read-only schema called `INFORMATION_SCHEMA` — you didn't create it, Snowflake did. Think of it as a **real-time data dictionary**: it reflects the current state of every object in that database as soon as the object is created, altered, or dropped. No lag.
>
> You'll also encounter **`SNOWFLAKE.ACCOUNT_USAGE`** later in the course — it covers the whole account (all databases, users, query history) but has a short latency window. `INFORMATION_SCHEMA` is per-database and instantaneous; `ACCOUNT_USAGE` is account-wide and historical. Use `INFORMATION_SCHEMA` for live governance queries like this one.
>
> `SHOW TABLES` gives a conversational list — good for a quick look. `INFORMATION_SCHEMA.TABLES` is queryable with SQL — better for inventories, reports, and anything you need to filter or aggregate. See the [INFORMATION_SCHEMA reference](https://docs.snowflake.com/en/sql-reference/info-schema).
>
> 💡 **Heads up — views appear here too.** `INFORMATION_SCHEMA.TABLES` lists views as well as tables. A view stores no rows, so its `row_count` and `bytes` come back `NULL` — that's why `SALES_BY_STORE` (a view) shows blank counts in the output below.

In [ ]:
-- Object sizes — what's in the shared schema and how large?
SELECT table_name,
       row_count,
       ROUND(bytes / 1024 / 1024, 2) AS size_mb
FROM PLATPLUS_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'ROASTERY_SALES'
ORDER BY bytes DESC NULLS LAST;

In [ ]:
-- Column inventory — spot anything that might need protection
SELECT table_name,
       column_name,
       data_type
FROM PLATPLUS_DB.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'ROASTERY_SALES'
ORDER BY table_name, ordinal_position;

> 🔎 **Scan the column names in the second output.** `FEEDBACK.COMMENT` is free-text customer data and `CUSTOMERS.FULL_NAME` is personally identifiable — both are the kind of columns that typically need masking before broader access is granted. The `CUSTOMER_ID` columns that link customers across tables are also worth flagging.
>
> **You've just produced your governance inventory.** In Lab 3 you'll use exactly this information to decide who gets SELECT on which tables and columns.

#### Step 2 · Know who can already see sensitive data

Before you decide who *should* have access, find out who *already does*. `SHOW GRANTS ON TABLE` returns every privilege granted on a specific object — regardless of which role you're running as.

> 💡 **Why this matters before Lab 3:** Admins commonly grant access to the tables they know about and overlook the ones they don't. Running `SHOW GRANTS ON TABLE` for every sensitive table is a quick check — a few seconds each — and prevents a compliance gap that can take weeks to remediate.

In [ ]:
-- Who has any privilege on the FEEDBACK table right now?
SHOW GRANTS ON TABLE PLATPLUS_DB.ROASTERY_SALES.FEEDBACK;

> 🔎 **Review the output.** Look at the `privilege`, `grantee_name`, and `granted_by` columns. You're looking for any role that has `SELECT` on `FEEDBACK` — that role can read customer comments. Is the access tighter or broader than you'd expect?
>
> In Lab 3 you'll design and apply a proper access model for these tables. This output is your baseline — you need to know the current state before you can change it.

#### Step 3 · Know your compute footprint

Warehouses bill by the second while running. A warehouse without `AUTO_SUSPEND` runs until manually stopped — the most common source of unexpected spend in new Snowflake accounts. Check what you have.

In [ ]:
-- Your warehouses — confirm AUTO_SUSPEND and AUTO_RESUME are configured
SHOW WAREHOUSES;

> 🔎 **Find your warehouse in the output** (look for the one named after your username). Check the `auto_suspend` and `auto_resume` columns. You created it with `AUTO_SUSPEND = 60` — that means it stops billing 60 seconds after its last query. `AUTO_RESUME = TRUE` means it restarts automatically when a query arrives. **This is correct governance.**
>
> 💡 **Admin rule of thumb:** Any warehouse in your account should have `AUTO_SUSPEND ≤ 300` and `AUTO_RESUME = TRUE`. Warehouses created without these settings are a cost risk worth flagging immediately. In the Credits & Cost lab you'll see how to monitor spend across all warehouses in the account.

#### Step 4 · Document what you create

An undocumented environment is an ungovernable one. [`COMMENT ON`](https://docs.snowflake.com/en/sql-reference/sql/comment) attaches a queryable description to any Snowflake object — databases, tables, schemas, columns. Anyone who looks at the account later can read it without running a data investigation.

In [ ]:
-- Attach comments to the objects you just created
COMMENT ON DATABASE {{my_db}}
  IS 'Personal lab workspace for Platform+ training — safe to archive after course.';
COMMENT ON TABLE {{my_db}}.SANDBOX.MY_TASTING_NOTES
  IS 'Test table created in Lab 1. Not production data.';

In [ ]:
-- Confirm the comment is stored
SHOW DATABASES LIKE '{{my_db}}';

> 🔎 **Find the `comment` column in the output.** You've attached a machine-readable, queryable label to your database. This pattern scales: `INFORMATION_SCHEMA.DATABASES` and `INFORMATION_SCHEMA.TABLES` both surface `comment` columns, making it possible to report on documentation coverage across the entire account — useful for governance audits.

#### Challenge · Retrieve your comment from INFORMATION_SCHEMA

You just added a comment via `COMMENT ON DATABASE`. Now prove it's stored by querying `INFORMATION_SCHEMA` directly — without using `SHOW DATABASES`.

Two blanks to fill in the cell below:
1. The **column name** that holds the description text you attached
2. The **view name** in `INFORMATION_SCHEMA` that lists databases

In [ ]:
-- CHALLENGE: surface the comment you attached to your database.
-- Fill in both blanks, then run the query.
--
-- Hint: INFORMATION_SCHEMA has a view that lists databases;
--       one of its columns stores the comment text.

SELECT database_name,
       ___           AS description    -- 1. which column holds the comment?
FROM   {{my_db}}.INFORMATION_SCHEMA.___  -- 2. which view lists databases?
ORDER BY database_name;

> 🔎 **Answer:** `comment` and `DATABASES`.
>
> ```sql
> SELECT database_name, comment AS description
> FROM   {{my_db}}.INFORMATION_SCHEMA.DATABASES
> ORDER BY database_name;
> ```
>
> `INFORMATION_SCHEMA.DATABASES` is queryable from any database context and shows every database your session can see. The `comment` column is live — it updates the moment you run `COMMENT ON`. This is the pattern you'd use to audit documentation coverage across all databases in the account.

## Check Your Work
Run the next cell to **check your work**. It queries your own database to confirm you created the objects and rows this lab asked for, and prints a ✅ / ⛔ result for each — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
checks = [
    ('Your SANDBOX.MY_TASTING_NOTES has at least 2 tasting rows', f'SELECT COUNT(*) FROM {my_db}.SANDBOX.MY_TASTING_NOTES', 2),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not found)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Knowledge check 🧠
Check your understanding with interactive quiz questions below. Each quiz cell has a widget that presents a multiple‑choice question about Snowflake functionality.

**Instructions:**
1. Hover your cursor over the notebook cell to reveal additional controls.
1. Click the ▶️ **Play button** on the right side of each quiz cell to run it.
1. Select your answer from the options provided.
1. Review the feedback before moving on.

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q4")

### Key takeaways. ❄️
- Context = role · warehouse · database · schema
- Hierarchy: account → database → schema → table/view
- You own objects you create in your own database; PLATPLUS_DB is read-only
- `INFORMATION_SCHEMA` is queryable with SQL — use it for inventories, not just `SHOW` commands
- `SHOW GRANTS ON TABLE` tells you who already has access — check it before you change any grants
- Every warehouse should have `AUTO_SUSPEND ≤ 300` and `AUTO_RESUME = TRUE`
- `COMMENT ON` is free governance — attach it to every object you create

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.